<a href="https://colab.research.google.com/github/MuskanZehra5/QML_Project_Diabetic_Retinopathy/blob/main/Model_Training_and_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch torchvision pandas numpy scikit-learn matplotlib seaborn tqdm pillow

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

DATASET_DIR = "/content/drive/MyDrive/MMRDR-CFP"

CSV_PATH = os.path.join(DATASET_DIR, "FP.csv")
IMAGE_DIR = os.path.join(DATASET_DIR, "img")

print("CSV exists:", os.path.exists(CSV_PATH))
print("Image folder exists:", os.path.exists(IMAGE_DIR))

CSV exists: True
Image folder exists: True


# Loading dataset

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head())

Dataset shape: (11118, 5)

Columns: ['type', 'image', 'grade', 'lesion', 'lr']


,type,image,grade,lesion,lr
0,FP,img/tr000001.jpg,0,"[0, 0, 0, 0, 0, 0, 0]",0
1,FP,img/tr000002.jpg,3,"[1, 1, 1, 0, 0, 0, 0]",1
2,FP,img/tr000003.jpg,2,"[1, 1, 1, 0, 0, 0, 0]",0
3,FP,img/tr000004.jpg,1,"[1, 0, 0, 0, 0, 0, 0]",1
4,FP,img/tr000005.jpg,0,"[0, 0, 0, 0, 0, 0, 0]",0


In [ ]:
## image count
images = os.listdir(IMAGE_DIR)

print("Total images:", len(images))
print("First 10 images:", images[:10])

Total images: 2893
First 10 images: ['tr008816.jpg', 'tr004385.jpg', 'ts001909.jpg', 'ts000008.jpg', 'tr005267.jpg', 'ts000582.jpg', 'tr008580.jpg', 'tr004632.jpg', 'tr000085.jpg', 'tr008553.jpg']


In [ ]:
import os
import pandas as pd

# Identify images that actually exist
df["split"] = df["image"].str.extract(r"(tr|ts)")

df["exists"] = df["image"].apply(
    lambda x: os.path.exists(os.path.join(DATASET_DIR, x))
)

available_df = df[df["exists"]].copy()

print("Available images:", len(available_df))

print("\nAvailable split distribution:")
print(available_df["split"].value_counts())

print("\nAvailable DR grade distribution:")
print(available_df["grade"].value_counts().sort_index())

print("\nGrade distribution by split:")
print(pd.crosstab(
    available_df["split"],
    available_df["grade"]
))

Available images: 2893

Available split distribution:
split
tr    2312
ts     581
Name: count, dtype: int64

Available DR grade distribution:
grade
0    1772
1     332
2     486
3     145
4     158
Name: count, dtype: int64

Grade distribution by split:
grade     0    1    2    3    4
split                          
tr     1417  272  378  117  128
ts      355   60  108   28   30


#Train, validation and test split

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os

# Keep only images that actually exist
available_df = available_df.copy()

# Separate official training and testing images
train_df = available_df[available_df["split"] == "tr"].copy()
test_df = available_df[available_df["split"] == "ts"].copy()

# Split training data into train and validation
train_split, val_split = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42,
    stratify=train_df["grade"]
)

# Reset indices
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training images:", len(train_split))
print("Validation images:", len(val_split))
print("Testing images:", len(test_df))

print("\nTraining grade distribution:")
print(train_split["grade"].value_counts().sort_index())

print("\nValidation grade distribution:")
print(val_split["grade"].value_counts().sort_index())

print("\nTesting grade distribution:")
print(test_df["grade"].value_counts().sort_index())

# Save split metadata
SPLIT_DIR = "/content/drive/MyDrive/MMRDR-CFP/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

train_split.to_csv(f"{SPLIT_DIR}/train.csv", index=False)
val_split.to_csv(f"{SPLIT_DIR}/validation.csv", index=False)
test_df.to_csv(f"{SPLIT_DIR}/test.csv", index=False)

print("\nSplit files saved successfully!")

Training images: 1849
Validation images: 463
Testing images: 581

Training grade distribution:
grade
0    1133
1     218
2     302
3      94
4     102
Name: count, dtype: int64

Validation grade distribution:
grade
0    284
1     54
2     76
3     23
4     26
Name: count, dtype: int64

Testing grade distribution:
grade
0    355
1     60
2    108
3     28
4     30
Name: count, dtype: int64

Split files saved successfully!


#Check and clean all three splits

In [ ]:
import os

def clean_split(df, split_name):
    df = df.copy()

    df["image_path"] = df["image"].apply(
        lambda x: os.path.join(DATASET_DIR, x)
    )

    missing = df[~df["image_path"].apply(os.path.isfile)]

    print(f"{split_name}:")
    print("  Original images:", len(df))
    print("  Missing images:", len(missing))
    if len(missing) > 0:
        print("  Examples:", missing["image"].head().tolist())

    df = df[df["image_path"].apply(os.path.isfile)].copy()
    df = df.drop(columns=["image_path"])
    return df.reset_index(drop=True)


train_split = clean_split(train_split, "Train")
val_split = clean_split(val_split, "Validation")
test_df = clean_split(test_df, "Test")

print("\nUpdated split sizes:")
print("Train:", len(train_split))
print("Validation:", len(val_split))
print("Test:", len(test_df))

Train:
  Original images: 1849
  Missing images: 0
Validation:
  Original images: 463
  Missing images: 0
Test:
  Original images: 581
  Missing images: 0

Updated split sizes:
Train: 1849
Validation: 463
Test: 581


#Create PyTorch Dataset and DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# ImageNet normalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class RetinalDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = os.path.join(DATASET_DIR, row["image"])

        image = Image.open(image_path).convert("RGB")
        label = int(row["grade"])

        if self.transform:
            image = self.transform(image)

        return image, label


# Create datasets
train_dataset = RetinalDataset(train_split, train_transform)
val_dataset = RetinalDataset(val_split, eval_transform)
test_dataset = RetinalDataset(test_df, eval_transform)

# Create loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Test one batch
images, labels = next(iter(train_loader))

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Train batches: 58
Validation batches: 15
Test batches: 19

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


#Initialize ResNet18 with an 8-dimensional bottleneck

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class DRModel(nn.Module):

    def __init__(self):
        super().__init__()

        # ImageNet-pretrained ResNet18
        self.backbone = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        # Remove original classification layer
        self.backbone.fc = nn.Identity()

        # Linear 8-dimensional bottleneck
        self.bottleneck = nn.Linear(512, 8)

        # Five-grade classification head
        self.classifier = nn.Linear(8, 5)

    def extract_features(self, x):
        x = self.backbone(x)
        x = self.bottleneck(x)
        return x

    def forward(self, x):
        features = self.extract_features(x)
        logits = self.classifier(features)
        return logits


model = DRModel().to(device)

print(model)
print("\nModel initialized successfully!")

DRModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, mo

#Train the model



In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import cohen_kappa_score
from collections import Counter

# Class weights to reduce the effect of class imbalance
grade_counts = train_split["grade"].value_counts().sort_index()

class_weights = 1.0 / torch.sqrt(
    torch.tensor(grade_counts.values, dtype=torch.float32)
)

class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Different learning rates for pretrained backbone and new layers
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 1e-5},
    {"params": model.bottleneck.parameters(), "lr": 1e-3},
    {"params": model.classifier.parameters(), "lr": 1e-3}
], weight_decay=1e-4)

EPOCHS = 5
best_val_loss = float("inf")

CHECKPOINT_PATH = os.path.join(DATASET_DIR, "best_resnet18.pth")

for epoch in range(EPOCHS):

    # ---------------- TRAINING ----------------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total

    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss = 0.0
    val_predictions = []
    val_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_dataset)

    val_qwk = cohen_kappa_score(
        val_labels,
        val_predictions,
        weights="quadratic"
    )

    val_accuracy = np.mean(
        np.array(val_predictions) == np.array(val_labels)
    )

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Training Loss:   {train_loss:.4f}")
    print(f"Training Acc:    {train_accuracy:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Acc:  {val_accuracy:.4f}")
    print(f"Validation QWK:  {val_qwk:.4f}")

    # Save best model using validation loss
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(model.state_dict(), CHECKPOINT_PATH)

        print("Best model checkpoint saved!")

print("\nTraining completed!")
print("Best validation loss:", best_val_loss)


Epoch 1/5
Training Loss:   1.2377
Training Acc:    0.6376
Validation Loss: 1.1106
Validation Acc:  0.6803
Validation QWK:  0.6587
Best model checkpoint saved!

Epoch 2/5
Training Loss:   0.9533
Training Acc:    0.7117
Validation Loss: 1.0235
Validation Acc:  0.7041
Validation QWK:  0.7167
Best model checkpoint saved!

Epoch 3/5
Training Loss:   0.8422
Training Acc:    0.7442
Validation Loss: 0.9703
Validation Acc:  0.6998
Validation QWK:  0.7696
Best model checkpoint saved!

Epoch 4/5
Training Loss:   0.7339
Training Acc:    0.7664
Validation Loss: 0.9521
Validation Acc:  0.7257
Validation QWK:  0.7762
Best model checkpoint saved!

Epoch 5/5
Training Loss:   0.6911
Training Acc:    0.7777
Validation Loss: 0.9636
Validation Acc:  0.7300
Validation QWK:  0.7637

Training completed!
Best validation loss: 0.9520709821008967


# PART 2

## download and load best checkpoint

In [ ]:
from google.colab import files

files.download(CHECKPOINT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### load the best Chcekpoint

In [ ]:
# Load the best-performing checkpoint
model.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location=device)
)

model.eval()

print("Best checkpoint loaded successfully!")

Best checkpoint loaded successfully!


###Create feature extraction DataLoaders

In [ ]:
# Use evaluation transforms for consistent feature extraction

train_eval_dataset = RetinalDataset(train_split, eval_transform)
val_eval_dataset = RetinalDataset(val_split, eval_transform)
test_eval_dataset = RetinalDataset(test_df, eval_transform)

train_eval_loader = DataLoader(
    train_eval_dataset, batch_size=32, shuffle=False, num_workers=0
)

val_eval_loader = DataLoader(
    val_eval_dataset, batch_size=32, shuffle=False, num_workers=0
)

test_eval_loader = DataLoader(
    test_eval_dataset, batch_size=32, shuffle=False, num_workers=0
)

print("Feature extraction loaders ready!")

Feature extraction loaders ready!


###Extract features

In [ ]:
import numpy as np

def extract_features(loader, model):

    model.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            features = model.extract_features(images)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    X = np.concatenate(all_features, axis=0)
    y = np.concatenate(all_labels, axis=0)

    return X, y


X_train, y_train = extract_features(train_eval_loader, model)

X_val, y_val = extract_features(val_eval_loader, model)

X_test, y_test = extract_features(test_eval_loader, model)


print("TRAIN FEATURES:", X_train.shape)
print("VALIDATION FEATURES:", X_val.shape)
print("TEST FEATURES:", X_test.shape)

TRAIN FEATURES: (1849, 8)
VALIDATION FEATURES: (463, 8)
TEST FEATURES: (581, 8)


###Save extracted features

In [ ]:
FEATURE_DIR = os.path.join(DATASET_DIR, "features")
os.makedirs(FEATURE_DIR, exist_ok=True)

np.savez(
    os.path.join(FEATURE_DIR, "dr_features.npz"),
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print("All extracted features saved successfully!")

All extracted features saved successfully!


#Part 2 — Classical ML Baselines

###Load the saved features and prepare evaluation metrics

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    roc_auc_score,
    cohen_kappa_score,
    roc_curve
)

# Primary binary target: referable DR (grade >= 2)
y_train_binary = (y_train >= 2).astype(int)
y_val_binary = (y_val >= 2).astype(int)
y_test_binary = (y_test >= 2).astype(int)

# Standardize using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Features standardized successfully!")

print("Training referable DR prevalence:",
      round(y_train_binary.mean(), 4))

print("Validation referable DR prevalence:",
      round(y_val_binary.mean(), 4))

print("Testing referable DR prevalence:",
      round(y_test_binary.mean(), 4))

Features standardized successfully!
Training referable DR prevalence: 0.2693
Validation referable DR prevalence: 0.27
Testing referable DR prevalence: 0.2857


###Create an evaluation function

In [ ]:
def calculate_ece(y_true, probabilities, bins=10):

    confidences = np.asarray(probabilities)
    ece = 0.0

    bin_edges = np.linspace(0, 1, bins + 1)

    for i in range(bins):

        if i == bins - 1:
            mask = (
                (confidences >= bin_edges[i]) &
                (confidences <= bin_edges[i + 1])
            )
        else:
            mask = (
                (confidences >= bin_edges[i]) &
                (confidences < bin_edges[i + 1])
            )

        if np.any(mask):

            accuracy = np.mean(y_true[mask])
            confidence = np.mean(confidences[mask])

            ece += np.mean(mask) * abs(accuracy - confidence)

    return ece


def evaluate_model(name, y_true, predictions, probabilities):

    y_binary = (y_true >= 2).astype(int)
    predicted_referable_probability = probabilities[:, 2:].sum(axis=1)

    auc = roc_auc_score(
        y_binary,
        predicted_referable_probability
    )

    qwk = cohen_kappa_score(
        y_true,
        predictions,
        weights="quadratic"
    )

    fpr, tpr, thresholds = roc_curve(
        y_binary,
        predicted_referable_probability
    )

    eligible = np.where(fpr <= 0.10)[0]
    sensitivity_90 = np.max(tpr[eligible])

    ece = calculate_ece(
        y_binary,
        predicted_referable_probability
    )

    return {
        "Model": name,
        "AUC": auc,
        "QWK": qwk,
        "Sensitivity@90%Specificity": sensitivity_90,
        "ECE": ece
    }

###Train B1: Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

lr_predictions = lr_model.predict(X_test_scaled)
lr_probabilities = lr_model.predict_proba(X_test_scaled)

print("Logistic Regression trained!")

Logistic Regression trained!


###Train B3: RBF-SVM

In [ ]:
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

svm_predictions = svm_model.predict(X_test_scaled)
svm_probabilities = svm_model.predict_proba(X_test_scaled)

print("RBF-SVM trained!")

RBF-SVM trained!


###Evaluate both models

In [ ]:
results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        lr_predictions,
        lr_probabilities
    )
)

results.append(
    evaluate_model(
        "RBF-SVM",
        y_test,
        svm_predictions,
        svm_probabilities
    )
)

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model      AUC      QWK  Sensitivity@90%Specificity      ECE
Logistic Regression 0.896690 0.746986                    0.632530 0.078267
            RBF-SVM 0.899085 0.745056                    0.644578 0.087296


What do these results mean?
- AUC ≈ 0.90: Both models distinguish referable DR (grades 2–4) from non-referable DR (grades 0–1) quite well on this subset.
- QWK ≈ 0.75: Both models show reasonably good agreement with the actual five-grade classifications.
- Sensitivity ≈ 0.64: At the selected operating point with no more than 10% false-positive rate, they identify approximately 64% of referable DR cases.
- ECE: Logistic Regression has slightly better calibration.
RBF-SVM has the highest AUC, but the difference is tiny. We cannot conclude that it is significantly better without confidence intervals.

###Prevalence baseline

In [ ]:
from sklearn.dummy import DummyClassifier

b0_model = DummyClassifier(strategy="prior")

b0_model.fit(X_train_scaled, y_train)

b0_predictions = b0_model.predict(X_test_scaled)
b0_probabilities = b0_model.predict_proba(X_test_scaled)

b0_results = evaluate_model(
    "Prevalence Baseline",
    y_test,
    b0_predictions,
    b0_probabilities
)

print(pd.DataFrame([b0_results]).to_string(index=False))

              Model  AUC  QWK  Sensitivity@90%Specificity     ECE
Prevalence Baseline  0.5  0.0                         0.0 0.01638


Run the backbone adequacy gate now

In [ ]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, cohen_kappa_score

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)

        all_probs.append(probabilities.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

# Probability of referable DR (grades 2, 3, 4)
referable_probs = all_probs[:, 2:].sum(axis=1)

# Binary AUC
binary_labels = (all_labels >= 2).astype(int)
backbone_auc = roc_auc_score(binary_labels, referable_probs)

# Five-class QWK
predictions = np.argmax(all_probs, axis=1)
backbone_qwk = cohen_kappa_score(
    all_labels,
    predictions,
    weights="quadratic"
)

print("BACKBONE ADEQUACY GATE")
print("----------------------")
print("Test AUC:", backbone_auc)
print("Test QWK:", backbone_qwk)
print("Gate passed:", backbone_auc >= 0.85)

BACKBONE ADEQUACY GATE
----------------------
Test AUC: 0.9070402090288866
Test QWK: 0.7369155659811271
Gate passed: True


#saving results

In [ ]:
import os
import pandas as pd
import numpy as np

RESULTS_DIR = os.path.join(DATASET_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Results folder:", RESULTS_DIR)

Results folder: /content/drive/MyDrive/MMRDR-CFP/results


In [ ]:
results = pd.DataFrame([
    {
        "Model": "Backbone (ResNet18)",
        "AUC": 0.907040,
        "QWK": 0.736916,
        "Sensitivity@90%Specificity": None,
        "ECE": None
    },
    {
        "Model": "B0 - Prevalence",
        "AUC": 0.500000,
        "QWK": 0.000000,
        "Sensitivity@90%Specificity": 0.000000,
        "ECE": 0.016380
    },
    {
        "Model": "B1 - Logistic Regression",
        "AUC": 0.896690,
        "QWK": 0.746986,
        "Sensitivity@90%Specificity": 0.632530,
        "ECE": 0.078267
    },
    {
        "Model": "B3 - RBF-SVM",
        "AUC": 0.899085,
        "QWK": 0.745056,
        "Sensitivity@90%Specificity": 0.644578,
        "ECE": 0.087296
    }
])

results.to_csv(
    os.path.join(RESULTS_DIR, "baseline_results.csv"),
    index=False
)

print(results)
print("\nResults saved successfully!")

                      Model       AUC       QWK  Sensitivity@90%Specificity  \
0       Backbone (ResNet18)  0.907040  0.736916                         NaN   
1           B0 - Prevalence  0.500000  0.000000                    0.000000   
2  B1 - Logistic Regression  0.896690  0.746986                    0.632530   
3              B3 - RBF-SVM  0.899085  0.745056                    0.644578   

        ECE  
0       NaN  
1  0.016380  
2  0.078267  
3  0.087296  

Results saved successfully!


In [ ]:
np.savez(
    os.path.join(RESULTS_DIR, "backbone_test_predictions.npz"),
    probabilities=all_probs,
    labels=all_labels
)

print("Backbone predictions saved!")

Backbone predictions saved!


In [ ]:
print("Files saved in results folder:")

for file in os.listdir(RESULTS_DIR):
    print("✓", file)

Files saved in results folder:
✓ baseline_results.csv
✓ backbone_test_predictions.npz
